# From Noise to Distributions: An Interactive Tour of GANs

This notebook builds a **Generative Adversarial Network (GAN)** from scratch and watches it learn - not on images, but on small **2D toy distributions** (a ring of 8 Gaussians) using tiny PyTorch MLPs that train in seconds on a CPU. Because everything is 2D, every claim we make is something you can *see and measure*, not just read.

## The big idea

A GAN learns to **sample** from a complex distribution you only have examples of. We tell the whole story in five parts:

1. **The generator is a distribution transformer** - it turns simple Gaussian noise `z` into a complex 2D distribution. (Why a *distribution*? Because many tasks have many valid answers.)
2. **The adversarial game** - a *discriminator* scores how real a point looks while the *generator* learns to fool it; the two co-evolve like predator and prey.
3. **The training algorithm** - alternate updating D and G, and watch the generated cloud slide onto the real modes, live.
4. **Why vanilla GANs are unstable** - the Jensen-Shannon non-overlap problem - and the **Wasserstein / WGAN** fix.
5. **How to evaluate generation** - quality, diversity, a 2D Frechet/FID analog, and the **mode-collapse** failure.

## Demo philosophy

Everything runs on synthetic 2D data with plain PyTorch - no images, no datasets, no GPU. The whole notebook executes end-to-end in well under a minute, and each interactive cell also renders a useful static view, so it stays useful even if widgets fail to load.

In [ ]:
# Environment setup - every library below ships with Colab, so no install is needed.
# Fallback only if ipywidgets is somehow missing:
# !pip install -q ipywidgets

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time

# ipywidgets is optional at import time: if it fails, the notebook still runs its
# static fallbacks (every interactive cell renders a default view without widgets).
try:
    import ipywidgets as widgets
    from IPython.display import display
    _WIDGETS_OK = True
except ImportError:
    widgets = None
    _WIDGETS_OK = False
    print('ipywidgets not available - interactive cells will show static fallbacks only.')

# Inline plotting (guarded so the cell also runs as a plain script).
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Reproducibility - seed every RNG so toy data, clouds, curves and metrics repeat on Run All.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device: CPU is intended and sufficient - every model here is a tiny MLP. We never force CUDA.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Matplotlib + style defaults shared by every later plot.
plt.rcParams['figure.figsize'] = (5, 5)
plt.rcParams['figure.dpi'] = 100
POINT_SIZE = 8
REAL_COLOR = 'tab:blue'
FAKE_COLOR = 'tab:red'
CMAP = 'RdBu'
AX_LIM = (-2.5, 2.5)   # fixed 2D window reused by every scatter / heatmap

# --- confirmation + sanity checks ---
assert device.type in ('cpu', 'cuda')
assert isinstance(AX_LIM, tuple) and AX_LIM[0] < AX_LIM[1]
print('Using device:', device, '| torch', torch.__version__, '| SEED', SEED)
print('torch.rand(2) =', torch.rand(2).tolist())       # eyeball: identical across re-runs
print('np.random.rand(2) =', np.random.rand(2).tolist())

In [ ]:
# Shared toy-GAN toolkit - defined once, reused by every concept below.

LATENT_DIM = 2  # default latent size (z ~ N(0, I))

def mode_centers(name='8gaussians'):
    '''True Gaussian mode centers for a dataset as an (M,2) array (or None if continuous).'''
    if name == '8gaussians':
        angles = np.arange(8) * (np.pi / 4.0)
        return np.stack([2.0 * np.cos(angles), 2.0 * np.sin(angles)], axis=1).astype(np.float32)
    elif name == '25gaussians':
        xs = np.linspace(-2.0, 2.0, 5)
        return np.array([[x, y] for x in xs for y in xs], dtype=np.float32)
    elif name in ('circle', 'two_moons'):
        return None  # continuous shapes: no discrete modes for coverage scoring
    else:
        raise ValueError('unknown dataset name: ' + str(name))

def sample_real_data(n, name='8gaussians'):
    '''Return n samples (float32 torch tensor on device) from the named 2D target.'''
    if name == '8gaussians':
        centers = mode_centers('8gaussians')
        idx = np.random.randint(0, len(centers), size=n)
        pts = centers[idx] + np.random.randn(n, 2).astype(np.float32) * 0.1
    elif name == '25gaussians':
        centers = mode_centers('25gaussians')
        idx = np.random.randint(0, len(centers), size=n)
        pts = centers[idx] + np.random.randn(n, 2).astype(np.float32) * 0.05
    elif name == 'circle':
        theta = np.random.uniform(0, 2 * np.pi, size=n)
        r = 1.8 + np.random.randn(n).astype(np.float32) * 0.05
        pts = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1).astype(np.float32)
    elif name == 'two_moons':
        n1 = n // 2; n2 = n - n1
        t1 = np.random.uniform(0, np.pi, size=n1)
        moon1 = np.stack([np.cos(t1), np.sin(t1)], axis=1)
        t2 = np.random.uniform(0, np.pi, size=n2)
        moon2 = np.stack([1.0 - np.cos(t2), 0.5 - np.sin(t2)], axis=1)
        pts = np.concatenate([moon1, moon2], axis=0).astype(np.float32)
        pts = pts + np.random.randn(n, 2).astype(np.float32) * 0.05
    else:
        raise ValueError('unknown dataset name: ' + str(name))
    return torch.from_numpy(pts.astype(np.float32)).to(device)

def sample_latent(n, latent_dim=LATENT_DIM):
    '''Standard-normal latent z of shape (n, latent_dim) on device.'''
    return torch.randn(n, latent_dim, device=device)

def make_generator(latent_dim, hidden=64, out_dim=2):
    '''Small MLP mapping z -> 2D point.'''
    return nn.Sequential(
        nn.Linear(latent_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, out_dim),
    ).to(device)

def make_discriminator(in_dim=2, hidden=64):
    '''Small MLP -> single RAW scalar (logit). Serves BCE-with-logits AND the WGAN critic.'''
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, 1),
    ).to(device)

def _to_np(x):
    '''Tensor or array -> detached cpu numpy (so matplotlib never sees a grad/cuda tensor).'''
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def scatter_distributions(real, fake, ax=None, title=''):
    '''Overlay real (blue) vs fake (red) point clouds on the fixed axes.'''
    if ax is None:
        _, ax = plt.subplots()
    r, f = _to_np(real), _to_np(fake)
    ax.scatter(f[:, 0], f[:, 1], s=POINT_SIZE, c=FAKE_COLOR, alpha=0.5, label='generated')
    ax.scatter(r[:, 0], r[:, 1], s=POINT_SIZE, c=REAL_COLOR, alpha=0.5, label='real')
    ax.set_xlim(*AX_LIM); ax.set_ylim(*AX_LIM); ax.set_aspect('equal')
    ax.set_title(title); ax.legend(loc='upper right', fontsize=7)
    return ax

def plot_decision_surface(D, ax=None, n_grid=120):
    '''Render D's realness field (sigmoid of the logit, display only) as a heatmap.'''
    if ax is None:
        _, ax = plt.subplots()
    lo, hi = AX_LIM
    XX, YY = np.meshgrid(np.linspace(lo, hi, n_grid), np.linspace(lo, hi, n_grid))
    grid = np.stack([XX.ravel(), YY.ravel()], axis=1).astype(np.float32)
    grid_t = torch.from_numpy(grid).to(device)
    was_training = D.training
    D.eval()
    with torch.no_grad():
        # sigmoid squashes the raw logit to a 0..1 realness field; for a WGAN critic this is
        # a monotone display transform only.
        field = torch.sigmoid(D(grid_t)).cpu().numpy().reshape(n_grid, n_grid)
    if was_training:
        D.train()
    im = ax.contourf(XX, YY, field, levels=20, cmap=CMAP, alpha=0.85)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
    return ax

# --- confirmation + sanity checks ---
print(f'device={device}, LATENT_DIM={LATENT_DIM}')
assert sample_real_data(256, '8gaussians').shape == (256, 2)
assert torch.isfinite(sample_real_data(256)).all()
_g = make_generator(LATENT_DIM); _out = _g(sample_latent(64))
assert _out.shape == (64, 2) and torch.isfinite(_out).all()
assert make_discriminator()(sample_real_data(16)).shape == (16, 1)
assert mode_centers('8gaussians').shape == (8, 2) and mode_centers('25gaussians').shape == (25, 2)
print('toolkit OK: data samplers, model builders and plotting helpers are ready')

## Concept 1 - The generator as a distribution transformer

A generator `G` takes a sample `z` from a **simple, known** distribution (here a 2D standard Normal) and outputs a point in a **complex** target distribution. As `z` ranges over *its* whole distribution, the outputs `G(z)` trace out the generator's output distribution `P_G`.

**Why do we want a distribution instead of a single output?** For tasks with *many valid answers* - predicting the next video frame (the car could turn left *or* right), drawing, dialogue - a deterministic regressor averages the conflicting answers into something blurry or invalid. A *stochastic* generator fed a sampled `z` can instead represent many plausible outputs at once.

In the next cell we literally push Gaussian noise through an **untrained** MLP generator and scatter its 2D outputs, so you can *see* simple noise being reshaped into a structured distribution - right next to the real target we will later learn to match.

In [ ]:
# Concept 1 made visible: push Gaussian noise through an UNTRAINED generator.
# This generator is never trained here; C08/C09 reuse it as the frozen 'fake' source.
untrained_generator = make_generator(LATENT_DIM)

N_VIS = 2000
with torch.no_grad():
    z = sample_latent(N_VIS)
    fake = untrained_generator(z)
real = sample_real_data(N_VIS)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: the noise-to-distribution map z -> G(z), next to the real target.
scatter_distributions(real, fake, ax=axes[0], title='untrained G:  z -> G(z)')

# Panel B (smoothness): color each output point by a latent coordinate z[:,0].
z_np, fake_np = _to_np(z), _to_np(fake)
sc = axes[1].scatter(fake_np[:, 0], fake_np[:, 1], s=POINT_SIZE, c=z_np[:, 0], cmap='viridis')
plt.colorbar(sc, ax=axes[1], fraction=0.046, pad=0.04, label='latent z[:,0]')
axes[1].set_xlim(*AX_LIM); axes[1].set_ylim(*AX_LIM); axes[1].set_aspect('equal')
axes[1].set_title('output colored by z[:,0] (neighbors stay neighbors)')

# Panel C (latent interpolation): a straight line in z maps to a continuous output path.
K = 50
torch.manual_seed(SEED)                      # fixed anchors -> reproducible path
z0 = torch.randn(LATENT_DIM, device=device)
z1 = torch.randn(LATENT_DIM, device=device)
t = torch.linspace(0, 1, K, device=device).unsqueeze(1)   # (K,1)
z_path = (1 - t) * z0 + t * z1                              # (K, LATENT_DIM)
assert z_path.shape == (K, LATENT_DIM)
with torch.no_grad():
    out_path = _to_np(untrained_generator(z_path))
axes[2].scatter(fake_np[:, 0], fake_np[:, 1], s=POINT_SIZE, c='lightgray', alpha=0.4)
axes[2].plot(out_path[:, 0], out_path[:, 1], '-o', c='crimson', ms=3, lw=1.2)
axes[2].set_xlim(*AX_LIM); axes[2].set_ylim(*AX_LIM); axes[2].set_aspect('equal')
axes[2].set_title('latent interpolation z0 -> z1')

plt.tight_layout(); plt.show()

assert untrained_generator(sample_latent(8)).shape == (8, 2)
assert torch.isfinite(fake).all()
assert out_path.shape[0] == K
print(f'generated cloud shape = {tuple(fake.shape)} (UNTRAINED - only visualizing z -> G(z), nothing learned yet)')

In [ ]:
# Interactive generator explorer: vary latent_dim, sample count, and reroll the random weights.
# A FRESH generator is built every call, so the cloud is a (random) distribution transformer.

def redraw_generator(latent_dim=LATENT_DIM, n_samples=1500, reroll=False):
    try:
        latent_dim, n_samples = int(latent_dim), int(n_samples)
        g = make_generator(latent_dim)            # fresh weights every call (reroll)
        with torch.no_grad():
            fake = g(sample_latent(n_samples, latent_dim))
        real = sample_real_data(n_samples)
        fig, ax = plt.subplots()
        scatter_distributions(real, fake, ax=ax,
                              title=f'random G | latent_dim={latent_dim}, n={n_samples}')
        plt.show()
    except Exception as e:
        print('redraw error:', e)

if _WIDGETS_OK:
    _ld = widgets.IntSlider(min=1, max=8, value=LATENT_DIM, description='latent_dim')
    _ns = widgets.IntSlider(min=200, max=4000, step=200, value=1500, description='n_samples')
    _rr = widgets.ToggleButton(value=False, description='reroll weights')
    generator_widget = widgets.interactive_output(
        redraw_generator, {'latent_dim': _ld, 'n_samples': _ns, 'reroll': _rr})
    display(widgets.VBox([widgets.HBox([_ld, _ns, _rr]), generator_widget]))
else:
    generator_widget = None
    redraw_generator(LATENT_DIM, 1500, False)   # static fallback if widgets are unavailable

# Static-path check: a direct call must never raise.
redraw_generator(LATENT_DIM, 800, False)

## Concept 2 - The adversarial game

Meet the **discriminator** `D`: a network that takes a 2D point and outputs a single **realness score** - large for points that look real, small for generated ones. The generator's job is to produce samples that make `D` output a *high* score, i.e. to **fool** it. In turn, `D` is retrained to tell real from generated.

Because each network improves in response to the other, they **co-evolve like predator and prey** - that is where the word *adversarial* comes from ("written as enemies, read as friends").

They are not yet trained jointly. In the next cell we render a single discriminator's **scalar field** over the plane as a heatmap, then run a few *discriminator-only* updates and watch the decision surface tighten around the real data - the discriminator's first move in the game.

In [ ]:
# Concept 2 made visible: the discriminator as a scalar realness field + Step-1-only training.
# Generator is FROZEN (untrained_generator from C05); only D learns here.

discriminator = make_discriminator()
d_opt = optim.Adam(discriminator.parameters(), lr=1e-3, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()   # raw logits in -> no log(0) blowups

N = 1024
real = sample_real_data(N, '8gaussians')
with torch.no_grad():
    fake = untrained_generator(sample_latent(N)).detach()   # frozen G, no gradient to G
ones = torch.ones(N, 1, device=device)
zeros = torch.zeros(N, 1, device=device)

def _mean_scores(D, real, fake):
    D.eval()
    with torch.no_grad():
        rs, fs = D(real).mean().item(), D(fake).mean().item()
    D.train()
    return rs, fs

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# BEFORE: untrained D - real and fake scores overlap, surface is undecided.
real_score_before, fake_score_before = _mean_scores(discriminator, real, fake)
plot_decision_surface(discriminator, ax=axes[0])
scatter_distributions(real, fake, ax=axes[0],
                      title=f'D before (real={real_score_before:.2f}, fake={fake_score_before:.2f})')

# Step 1 ONLY: train D to label real=1, fake=0 (generator frozen).
D_STEPS = 150
losses = []
for _ in range(D_STEPS):
    d_opt.zero_grad()
    loss = bce(discriminator(real), ones) + bce(discriminator(fake), zeros)
    loss.backward(); d_opt.step()
    losses.append(loss.item())

# AFTER: the surface sharpens into a boundary hugging the real ring.
real_score_after, fake_score_after = _mean_scores(discriminator, real, fake)
plot_decision_surface(discriminator, ax=axes[1])
scatter_distributions(real, fake, ax=axes[1],
                      title=f'D after {D_STEPS} steps (real={real_score_after:.2f}, fake={fake_score_after:.2f})')
plt.tight_layout(); plt.show()
decision_surface_figure = fig

print(f'before: gap (real-fake) = {real_score_before - fake_score_before:.3f}')
print(f'after : gap (real-fake) = {real_score_after - fake_score_after:.3f}')
assert all(np.isfinite([real_score_before, fake_score_before, real_score_after, fake_score_after]))
assert (real_score_after - fake_score_after) > (real_score_before - fake_score_before)
assert losses[-1] < losses[0] + 1e-6
print('This is only Step 1: the generator has NOT moved yet.')

In [ ]:
# Interactive: how many D-only steps does it take to 'win' against a frozen generator?
N = 1024

def redraw_discriminator(n_steps=120):
    try:
        n_steps = int(n_steps)
        D = make_discriminator()                              # fresh D each call
        opt = optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))
        real = sample_real_data(N, '8gaussians')
        with torch.no_grad():
            fake = untrained_generator(sample_latent(N)).detach()
        ones = torch.ones(N, 1, device=device); zeros = torch.zeros(N, 1, device=device)
        for _ in range(n_steps):                              # n_steps==0 -> flat surface
            opt.zero_grad()
            loss = bce(D(real), ones) + bce(D(fake), zeros)
            loss.backward(); opt.step()
        D.eval()
        with torch.no_grad():
            rs, fs = D(real).mean().item(), D(fake).mean().item()
        fig, ax = plt.subplots()
        plot_decision_surface(D, ax=ax)
        scatter_distributions(real, fake, ax=ax, title=f'D after {n_steps} steps | gap={rs - fs:.2f}')
        plt.show()
        print(f'steps={n_steps}: mean real={rs:.3f}, mean fake={fs:.3f}, gap={rs - fs:.3f}')
    except Exception as e:
        print('redraw error:', e)

if _WIDGETS_OK:
    _ss = widgets.IntSlider(min=0, max=400, step=10, value=120, description='D-only steps')
    discriminator_widget = widgets.interactive_output(redraw_discriminator, {'n_steps': _ss})
    display(widgets.VBox([_ss, discriminator_widget]))
else:
    discriminator_widget = None
    redraw_discriminator(120)   # static fallback

# Static-path check.
redraw_discriminator(120)

## Concept 3 - The training algorithm

Initialize `G` and `D`, then repeat two **alternating** steps every iteration:

- **Step 1 (fix G, update D):** separate real from generated - real labelled `1`, generated labelled `0` - with binary cross-entropy.
- **Step 2 (fix D, update G):** make `D` assign **high** scores to `G`'s samples. The generator's gradient flows *through the frozen* `D`.

Iterating this adversarial loop is exactly what drives the generated distribution toward the real one. The lecture's anime-face quality climbing from 100 to 50,000 updates is, on our 2D toy data, the generated point cloud **sliding onto the real modes**.

In the next cell we implement this as `train_gan` and snapshot the generated distribution at several milestones, plotting the G/D loss curves alongside - so convergence is *watched*, not asserted.

In [ ]:
# Concept 3: the alternating GAN training loop. train_gan is the ONE training implementation;
# its mode flag selects vanilla GAN (BCE) vs WGAN (critic + weight clipping). Reused by C12/C14/C15/C17.

def train_gan(name='8gaussians', mode='gan', latent_dim=LATENT_DIM, lr=2e-4,
              d_steps=1, n_iters=5000, clip=0.01,
              snapshot_iters=(0, 500, 2000, 5000), batch_size=256):
    if mode not in ('gan', 'wgan'):
        raise ValueError('mode must be gan or wgan, got ' + repr(mode))
    assert isinstance(n_iters, int) and n_iters > 0
    snap = set(min(int(s), n_iters) for s in snapshot_iters)

    G = make_generator(latent_dim)
    D = make_discriminator()
    g_opt = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    d_opt = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce_loss = nn.BCEWithLogitsLoss()
    ones = torch.ones(batch_size, 1, device=device)
    zeros = torch.zeros(batch_size, 1, device=device)
    g_losses, d_losses, snapshots = [], [], {}

    def _snap(it):
        with torch.no_grad():
            snapshots[it] = _to_np(G(sample_latent(2000, latent_dim)))

    if 0 in snap:
        _snap(0)

    for it in range(1, n_iters + 1):
        # ---- STEP 1: update D (generator frozen) ----
        for _ in range(d_steps):
            real = sample_real_data(batch_size, name)
            with torch.no_grad():
                fake = G(sample_latent(batch_size, latent_dim))   # detached: no grad to G
            if mode == 'gan':
                d_loss = bce_loss(D(real), ones) + bce_loss(D(fake), zeros)
            else:  # wgan critic maximizes E[D(real)] - E[D(fake)] => minimize the negative
                d_loss = -(D(real).mean() - D(fake).mean())
            d_opt.zero_grad(); d_loss.backward(); d_opt.step()
            if mode == 'wgan':
                for p in D.parameters():
                    p.data.clamp_(-clip, clip)   # 1-Lipschitz via weight clipping
        # ---- STEP 2: update G once (D frozen; gradient flows through D) ----
        fake = G(sample_latent(batch_size, latent_dim))
        if mode == 'gan':
            g_loss = bce_loss(D(fake), ones)   # non-saturating: push D(fake) high
        else:
            g_loss = -D(fake).mean()
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

        d_losses.append(float(d_loss.item())); g_losses.append(float(g_loss.item()))
        if not (np.isfinite(d_losses[-1]) and np.isfinite(g_losses[-1])):
            print(f'[warn] non-finite loss at iter {it} - stopping early (try a lower lr).')
            break
        if it in snap:
            _snap(it)

    return {'G': G, 'D': D, 'g_losses': g_losses, 'd_losses': d_losses, 'snapshots': snapshots}

def _count_modes(fake_np, centers, radius=0.5):
    '''How many mode centers have at least one generated point within radius.'''
    if centers is None:
        return 0
    d = np.linalg.norm(fake_np[:, None, :] - centers[None, :, :], axis=2)  # (n, M)
    return int((d.min(axis=0) <= radius).sum())

# --- run the default vanilla GAN on the 8-Gaussians ring ---
result = train_gan(name='8gaussians', mode='gan', n_iters=5000, snapshot_iters=(0, 500, 2000, 5000))
trained_generator = result['G']

snap_keys = sorted(result['snapshots'].keys())
real = sample_real_data(2000, '8gaussians')
fig, axes = plt.subplots(1, len(snap_keys), figsize=(4 * len(snap_keys), 4))
if len(snap_keys) == 1:
    axes = [axes]
for ax, it in zip(axes, snap_keys):
    scatter_distributions(real, result['snapshots'][it], ax=ax, title=f'iter {it}')
plt.tight_layout(); plt.show()
gan_snapshots_figure = fig

fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.plot(result['g_losses'], label='G loss', alpha=0.8)
ax2.plot(result['d_losses'], label='D loss', alpha=0.8)
ax2.set_xlabel('iteration'); ax2.set_ylabel('loss'); ax2.set_title('GAN training losses'); ax2.legend()
plt.tight_layout(); plt.show()
gan_loss_figure = fig2

centers = mode_centers('8gaussians')
n_covered = _count_modes(result['snapshots'][snap_keys[-1]], centers, radius=0.5)
print(f'final mode coverage: {n_covered}/8 modes reached (radius=0.5)')

assert len(result['g_losses']) == len(result['d_losses'])
assert all(np.isfinite(v).all() for v in result['snapshots'].values())
assert trained_generator(sample_latent(16)).shape == (16, 2)

In [ ]:
# Interactive trainer: feel the GAN's sensitivity to lr, d_steps, latent_dim, dataset.
N_ITERS_CAP = 4000   # hard cap so Run All stays fast and the kernel cannot hang

def run_training(lr=2e-4, d_steps=1, latent_dim=LATENT_DIM, n_iters=2000, dataset='8gaussians'):
    try:
        lr = float(lr); d_steps = int(d_steps); latent_dim = int(latent_dim)
        n_iters = min(int(n_iters), N_ITERS_CAP)
        res = train_gan(name=dataset, mode='gan', lr=lr, d_steps=d_steps,
                        latent_dim=latent_dim, n_iters=n_iters, snapshot_iters=(n_iters,))
        real = sample_real_data(2000, dataset)
        fake = res['snapshots'][n_iters]
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        scatter_distributions(real, fake, ax=axes[0],
                              title=f'{dataset} | lr={lr:.1e}, d_steps={d_steps}, iters={n_iters}')
        axes[1].plot(res['g_losses'], label='G loss', alpha=0.8)
        axes[1].plot(res['d_losses'], label='D loss', alpha=0.8)
        axes[1].set_title('losses'); axes[1].set_xlabel('iteration'); axes[1].legend()
        plt.tight_layout(); plt.show()
        c = mode_centers(dataset)
        cov = _count_modes(fake, c, 0.5) if c is not None else None
        print('final mode coverage:', f'{cov}/{len(c)}' if cov is not None else 'n/a (continuous dataset)')
    except Exception as e:
        print('training run error (try a lower lr if it diverged):', e)

if _WIDGETS_OK:
    _lr = widgets.FloatLogSlider(base=10, min=-5, max=-2, value=2e-4, step=0.1, description='lr')
    _ds = widgets.IntSlider(min=1, max=5, value=1, description='d_steps')
    _ld = widgets.IntSlider(min=1, max=8, value=LATENT_DIM, description='latent_dim')
    _ni = widgets.IntSlider(min=200, max=N_ITERS_CAP, step=200, value=2000, description='n_iters')
    _dd = widgets.Dropdown(options=['8gaussians', '25gaussians', 'circle'], value='8gaussians', description='dataset')
    training_widget = widgets.interact_manual(run_training, lr=_lr, d_steps=_ds,
                                              latent_dim=_ld, n_iters=_ni, dataset=_dd)
else:
    training_widget = None

# Static default (balanced settings) so a converged result always shows.
run_training(2e-4, 1, LATENT_DIM, 2000, '8gaussians')

## Concept 4 - Why vanilla GANs are unstable, and the Wasserstein fix

The generator is implicitly minimizing a **divergence** between `P_G` and `P_data`, and the optimal discriminator's objective corresponds to the **Jensen-Shannon (JS) divergence**.

**The problem:** real and generated data lie on low-dimensional manifolds that, especially early in training, **barely overlap**. Whenever two distributions don't overlap, the JS divergence is stuck at the constant `log 2` - giving **zero useful gradient**. The generator gets no signal about which way to move.

**The fix:** the **Wasserstein (earth-mover) distance** measures the minimal cost to transport one distribution onto the other, so it varies **smoothly** with their separation *even when they are disjoint*. WGAN therefore replaces the JS game with maximizing `E[D(real)] - E[D(fake)]` over a **1-Lipschitz critic** `D`, enforced in the original WGAN by **clipping** the critic's weights to `[-c, c]` (later work uses a gradient penalty or spectral normalization).

Next we plot JS vs Wasserstein against the gap between two distributions to expose the flat-gradient problem, then compare vanilla-GAN vs WGAN training on the same toy data.

In [ ]:
# Concept 4, Part A: WHY vanilla GANs stall - JS saturates at log(2) when supports separate,
# while the Wasserstein (earth-mover) distance keeps a usable gradient.

def js_divergence_1d(samples_p, samples_q, bins=60, lo=-6.0, hi=6.0):
    p_hist, _ = np.histogram(_to_np(samples_p).ravel(), bins=bins, range=(lo, hi))
    q_hist, _ = np.histogram(_to_np(samples_q).ravel(), bins=bins, range=(lo, hi))
    eps = 1e-10
    p = p_hist.astype(np.float64) + eps; p /= p.sum()
    q = q_hist.astype(np.float64) + eps; q /= q.sum()
    m = 0.5 * (p + q)
    kl = lambda a, b: np.sum(a * np.log(a / b))
    return float(0.5 * kl(p, m) + 0.5 * kl(q, m))   # natural log -> ceiling is log(2)

def wasserstein_1d(samples_p, samples_q):
    a = np.sort(_to_np(samples_p).ravel())
    b = np.sort(_to_np(samples_q).ravel())
    n = min(len(a), len(b))                          # subsample longer to equal length
    a = a[np.linspace(0, len(a) - 1, n).astype(int)]
    b = b[np.linspace(0, len(b) - 1, n).astype(int)]
    return float(np.mean(np.abs(a - b)))             # exact W1 for equal-size 1D empiricals

deltas = np.linspace(0, 6, 25)
M = 4000
base = np.random.randn(M)
js_vals, w_vals = [], []
for delta in deltas:
    shifted = np.random.randn(M) + delta
    js_vals.append(js_divergence_1d(base, shifted, lo=-6.0, hi=14.0))
    w_vals.append(wasserstein_1d(base, shifted))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(deltas, js_vals, '-o', ms=3, label='JS divergence')
ax.plot(deltas, w_vals, '-s', ms=3, label='Wasserstein-1')
ax.axhline(np.log(2), ls='--', c='gray', label='log(2) (JS ceiling)')
ax.set_xlabel('separation between the two distributions')
ax.set_ylabel('divergence / distance')
ax.set_title('JS saturates (flat -> no gradient); Wasserstein grows smoothly')
ax.legend(); plt.tight_layout(); plt.show()
divergence_curve_figure = fig

# Part B: vanilla GAN vs WGAN on the SAME 8-Gaussians data, matched budget.
NB = 2000
gan = train_gan(name='8gaussians', mode='gan', n_iters=NB, snapshot_iters=(NB,))
wgan = train_gan(name='8gaussians', mode='wgan', clip=0.01, d_steps=5, n_iters=NB, snapshot_iters=(NB,))

real = sample_real_data(2000, '8gaussians')
centers = mode_centers('8gaussians')
fig2, axes = plt.subplots(2, 2, figsize=(11, 9))
scatter_distributions(real, gan['snapshots'][NB], ax=axes[0, 0], title='vanilla GAN - final')
scatter_distributions(real, wgan['snapshots'][NB], ax=axes[0, 1], title='WGAN - final')
axes[1, 0].plot(gan['g_losses'], label='G'); axes[1, 0].plot(gan['d_losses'], label='D')
axes[1, 0].set_title('GAN losses (BCE)'); axes[1, 0].set_xlabel('iteration'); axes[1, 0].legend()
axes[1, 1].plot(wgan['g_losses'], label='G'); axes[1, 1].plot(wgan['d_losses'], label='critic')
axes[1, 1].set_title('WGAN losses (critic ~ -Wasserstein)'); axes[1, 1].set_xlabel('iteration'); axes[1, 1].legend()
plt.tight_layout(); plt.show()
gan_vs_wgan_figure = fig2

gan_cov = _count_modes(gan['snapshots'][NB], centers, 0.5)
wgan_cov = _count_modes(wgan['snapshots'][NB], centers, 0.5)
print(f'mode coverage - vanilla GAN: {gan_cov}/8   WGAN: {wgan_cov}/8')

# sanity
assert w_vals[-1] > w_vals[0]
sep_js = js_divergence_1d(0.2 * np.random.randn(M) - 4, 0.2 * np.random.randn(M) + 4, lo=-8.0, hi=8.0)
assert abs(sep_js - np.log(2)) < 0.05
assert js_divergence_1d(base, base, lo=-6.0, hi=14.0) < 1e-6

In [ ]:
# Interactive: flip GAN vs WGAN and sweep the WGAN weight-clip c.
N_ITERS_CAP = 4000

def run_objective(objective='wgan', clip=0.01, lr=2e-4, n_iters=2000):
    try:
        clip = float(clip); lr = float(lr); n_iters = min(int(n_iters), N_ITERS_CAP)
        d_steps = 5 if objective == 'wgan' else 1   # standard practice, not another slider
        res = train_gan(name='8gaussians', mode=objective, clip=clip, d_steps=d_steps,
                        lr=lr, n_iters=n_iters, snapshot_iters=(n_iters,))
        real = sample_real_data(2000, '8gaussians')
        fake = res['snapshots'][n_iters]
        clip_txt = f', clip={clip}' if objective == 'wgan' else ' (clip ignored for gan)'
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        scatter_distributions(real, fake, ax=axes[0],
                              title=f'{objective}{clip_txt}, lr={lr:.1e}, iters={n_iters}')
        axes[1].plot(res['g_losses'], label='G'); axes[1].plot(res['d_losses'], label='D/critic')
        axes[1].set_title('losses'); axes[1].set_xlabel('iteration'); axes[1].legend()
        plt.tight_layout(); plt.show()
        cov = _count_modes(fake, mode_centers('8gaussians'), 0.5)
        print(f'objective={objective}: final mode coverage {cov}/8')
    except Exception as e:
        print('run error:', e)

if _WIDGETS_OK:
    _obj = widgets.ToggleButtons(options=['gan', 'wgan'], value='wgan', description='objective')
    _clip = widgets.FloatSlider(min=0.005, max=0.1, step=0.005, value=0.01, description='WGAN clip c')
    _lr2 = widgets.FloatLogSlider(base=10, min=-5, max=-2, value=2e-4, step=0.1, description='lr')
    _ni2 = widgets.IntSlider(min=200, max=N_ITERS_CAP, step=200, value=2000, description='n_iters')
    wgan_widget = widgets.interact_manual(run_objective, objective=_obj, clip=_clip, lr=_lr2, n_iters=_ni2)
else:
    wgan_widget = None

# Static default (WGAN with a sensible clip).
run_objective('wgan', 0.01, 2e-4, 2000)

## Concept 5 - How to evaluate generative models

The **training loss does not tell you whether samples are good**. Generation is judged along two axes:

- **Quality** - each generated sample looks like real data. (With images, a pretrained classifier gives a *concentrated* class posterior `P(c|y)`.)
- **Diversity** - the generator covers the *whole* data distribution, not just part of it (a *uniform* marginal `P(c)` over modes).

**Inception Score** combines both; **FID** measures the Frechet distance between Gaussians fitted to real vs generated features in a pretrained network's feature space (smaller = better).

The signature failure is **mode collapse / mode dropping**: the generator produces only one or a few real modes - maximizing its fooling power while abandoning diversity. (We also don't want a generator that merely *memorizes* training points.)

We have no images, so we use a faithful 2D analog: a **Frechet/FID distance** between Gaussian fits of real vs generated points, plus a **mode-coverage count** over the known Gaussian centers - computed for a healthy vs a deliberately collapsed generator.

In [ ]:
# Concept 5: make generation measurable + expose mode collapse on the toy data.

def _matrix_sqrt_psd(S):
    '''PSD matrix square root via symmetric eigendecomposition (no scipy needed).'''
    S = 0.5 * (S + S.T)
    lam, V = np.linalg.eigh(S)
    lam = np.clip(lam, 0.0, None)
    return (V * np.sqrt(lam)) @ V.T

def frechet_distance_2d(real, fake):
    '''FID analog: ||mu_r-mu_f||^2 + Tr(C_r + C_f - 2 sqrtm(C_r C_f)). Lower is better.'''
    r, f = _to_np(real), _to_np(fake)
    mu_r, mu_f = r.mean(0), f.mean(0)
    eps = 1e-6
    C_r = np.cov(r, rowvar=False) + eps * np.eye(2)
    C_f = np.cov(f, rowvar=False) + eps * np.eye(2)
    A = _matrix_sqrt_psd(C_r)
    tr_sqrt = np.trace(_matrix_sqrt_psd(A @ C_f @ A))   # Tr(sqrtm(C_r C_f))
    fid = float(np.sum((mu_r - mu_f) ** 2) + np.trace(C_r) + np.trace(C_f) - 2.0 * tr_sqrt)
    return max(fid, 0.0)

def mode_coverage(fake, centers, radius=0.5):
    '''Return (n_modes_covered, per_mode_counts). A mode is covered if >=1 point within radius.'''
    if centers is None:
        print('mode_coverage: dataset has no discrete modes; returning (0, []).')
        return 0, np.array([])
    f = _to_np(fake)
    d = np.linalg.norm(f[:, None, :] - centers[None, :, :], axis=2)   # (n, M)
    nearest = d.argmin(axis=1)
    within = d.min(axis=1) <= radius
    per_mode = np.zeros(len(centers), dtype=int)
    for idx, ok in zip(nearest, within):
        if ok:
            per_mode[idx] += 1
    return int((per_mode > 0).sum()), per_mode

RADIUS = 0.5
centers = mode_centers('8gaussians')
real = sample_real_data(2000, '8gaussians')

# HEALTHY generator from C11.
with torch.no_grad():
    healthy_fake = _to_np(trained_generator(sample_latent(2000)))

# COLLAPSED generator: induce cheaply via a strong D/G imbalance + short training;
# fall back to a hand-built degenerate generator if that does not collapse enough.
try:
    collapse_res = train_gan(name='8gaussians', mode='gan', lr=5e-3, d_steps=5,
                            n_iters=400, snapshot_iters=(400,))
    collapsed_generator = collapse_res['G']
    with torch.no_grad():
        collapsed_fake = _to_np(collapsed_generator(sample_latent(2000)))
    if mode_coverage(collapsed_fake, centers, RADIUS)[0] > 4:
        raise RuntimeError('imbalanced run did not collapse enough; using degenerate fallback')
except Exception as e:
    print('collapse inducer fallback:', e)
    class _DegenerateG(nn.Module):
        '''Maps every z near a single mode -> guaranteed collapse.'''
        def __init__(self, center):
            super().__init__()
            self.center = torch.tensor(center, dtype=torch.float32, device=device)
        def forward(self, z):
            return self.center + 0.05 * torch.randn(z.shape[0], 2, device=device)
    collapsed_generator = _DegenerateG(centers[0])
    with torch.no_grad():
        collapsed_fake = _to_np(collapsed_generator(sample_latent(2000)))

# scatter healthy vs collapsed against the real ring
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
scatter_distributions(real, healthy_fake, ax=axes[0], title='healthy generator')
scatter_distributions(real, collapsed_fake, ax=axes[1], title='collapsed generator')
plt.tight_layout(); plt.show()

fid_h = frechet_distance_2d(real, healthy_fake)
fid_c = frechet_distance_2d(real, collapsed_fake)
cov_h, counts_h = mode_coverage(healthy_fake, centers, RADIUS)
cov_c, counts_c = mode_coverage(collapsed_fake, centers, RADIUS)
print(f'HEALTHY  : Frechet(2D) = {fid_h:.3f} | mode coverage = {cov_h}/8')
print(f'COLLAPSED: Frechet(2D) = {fid_c:.3f} | mode coverage = {cov_c}/8')
print('Lower Frechet and fuller / more uniform coverage = better generation.')

fig2, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(centers)); w = 0.4
ax.bar(x - w / 2, counts_h, width=w, label=f'healthy (cov {cov_h}/8)', color=REAL_COLOR)
ax.bar(x + w / 2, counts_c, width=w, label=f'collapsed (cov {cov_c}/8)', color=FAKE_COLOR)
ax.set_xlabel('mode index'); ax.set_ylabel('# generated samples'); ax.set_title('per-mode coverage')
ax.legend(); plt.tight_layout(); plt.show()
evaluation_figure = fig2

# sanity
real_resampled = sample_real_data(2000, '8gaussians')
assert frechet_distance_2d(real, real_resampled) < fid_c
assert (cov_c < cov_h) or (fid_c > fid_h)
assert np.isfinite(fid_h) and np.isfinite(fid_c) and fid_h >= 0 and fid_c >= 0
assert len(counts_h) == len(centers)

In [ ]:
# Interactive: dial from full diversity (alpha=0) to full collapse (alpha=1) and watch metrics move.
centers = mode_centers('8gaussians')
real_eval = sample_real_data(2000, '8gaussians')

# Fixed latent + a fixed replace-mask so ONLY alpha drives the change (no resampling noise).
z_fixed = sample_latent(2000)
with torch.no_grad():
    healthy_pts = _to_np(trained_generator(z_fixed))
    collapsed_pts = _to_np(collapsed_generator(z_fixed))
_mask_u = np.random.rand(healthy_pts.shape[0])

def redraw_eval(alpha=0.5):
    try:
        alpha = float(min(max(alpha, 0.0), 1.0))
        # replace a fraction alpha of healthy points by their collapsed counterparts:
        # the cloud literally shrinks toward the collapsed mode(s) as alpha rises.
        replace = (_mask_u < alpha)[:, None]
        blended = np.where(replace, collapsed_pts, healthy_pts)
        cov, counts = mode_coverage(blended, centers, RADIUS)
        fid = frechet_distance_2d(real_eval, blended)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        scatter_distributions(real_eval, blended, ax=axes[0], title=f'collapse alpha={alpha:.2f}')
        axes[1].bar(np.arange(len(centers)), counts, color=FAKE_COLOR)
        axes[1].set_xlabel('mode index'); axes[1].set_ylabel('# samples')
        axes[1].set_title(f'Frechet={fid:.2f} | coverage={cov}/8')
        plt.tight_layout(); plt.show()
        print(f'alpha={alpha:.2f}: Frechet(2D)={fid:.3f}, mode coverage={cov}/8')
    except Exception as e:
        print('redraw error:', e)

if _WIDGETS_OK:
    _alpha = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='collapse')
    evaluation_widget = widgets.interactive_output(redraw_eval, {'alpha': _alpha})
    display(widgets.VBox([_alpha, evaluation_widget]))
else:
    evaluation_widget = None
    redraw_eval(0.5)

# Static-path check (forward passes only, no retraining -> cannot hang the kernel).
redraw_eval(0.5)

## Wrap-up: the GAN story, start to finish

We built one continuous storyline on fast 2D toy data:

1. **Distribution transformer** - a generator turns simple Gaussian noise into a complex 2D distribution (Concept 1).
2. **The adversarial game** - a discriminator scores realness while the generator learns to fool it (Concept 2).
3. **The alternating algorithm** - `train_gan` migrates the generated cloud onto the real modes, watched live (Concept 3).
4. **JS non-overlap -> WGAN** - flat JS gradients explain GAN instability; the Wasserstein objective fixes it (Concept 4).
5. **Evaluation** - quality and diversity via a 2D Frechet/FID analog and mode coverage, with mode collapse as the key failure (Concept 5).

Every claim - convergence, the flat JS gradient, WGAN's added stability, and collapse - was **visualized and measured**, not asserted.

## Intentionally left out (great next steps)

To keep the notebook reliably runnable in under a minute, we skipped methods that need labeled / image data or much longer training:

- **Conditional GAN** (text-to-image, pix2pix)
- **CycleGAN** and cycle-consistency for unpaired translation
- The **discriminator-as-f-divergence** theory in full
- **Lipschitz enforcement beyond weight clipping** - gradient penalty (WGAN-GP), spectral normalization

Each is a natural follow-up notebook.